<a href="https://colab.research.google.com/github/anelchik/anelsinternshipwork/blob/main/w04_baseline_score_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

**Lane:** Content Review Priority Ranking  
**Decision:** Which pseudonymized content pages should an SEO specialist review first?  
**Claim limit:** This baseline identifies observable CTR opportunities; it does not prove search-intent mismatch or causality.


## 1. My rule and its reason codes

I rank pages higher when they have meaningful search visibility and their observed CTR is below the typical CTR of pages in a similar average-position bucket. Position adjustment matters because a page in position 2 should not be judged by the same raw CTR threshold as a page in position 30.

The two checked signals are:

1. **CTR versus average position** — a real FlyRank CTR-fix signal.
2. **Impression volume** — a measure of opportunity size.

The encoded rule is:

`baseline_action_score = 0.65 × CTR-gap percentile + 0.35 × impression percentile`

Pages with a positive CTR gap and at least median impression volume receive `REVIEW_CTR_CONTENT`; all others receive `MONITOR`.

Reason codes: `CTR_GAP_HIGH_VISIBILITY`, `CTR_GAP_ONLY`, `HIGH_VISIBILITY_ONLY`, and `NO_BASELINE_TRIGGER`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

CANDIDATE_PATHS = [
    Path("content_refresh_anonymized.csv"),
    Path("./content_refresh_anonymized.csv"),
    Path("../content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        f"Current working directory: {Path.cwd()}"
    )

df = pd.read_csv(data_path)
print(f"Loaded: {data_path.resolve()}")
print(f"Shape: {df.shape}")

required = {"content_id", "client_id", "impressions_90d", "ctr", "avg_position"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

work = df.copy()
for col in ["impressions_90d", "ctr", "avg_position"]:
    work[col] = pd.to_numeric(work[col], errors="coerce")

work = work.dropna(subset=["impressions_90d", "ctr", "avg_position"])
work = work[
    (work["impressions_90d"] >= 0)
    & (work["ctr"] >= 0)
    & (work["avg_position"] > 0)
].copy()

if work["ctr"].quantile(0.95) > 1:
    work["ctr"] = work["ctr"] / 100

work["position_bucket"] = pd.cut(
    work["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    include_lowest=True,
)

ctr_by_position = (
    work.groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)
ctr_by_position["median_ctr"] = ctr_by_position["median_ctr"].round(4)
ctr_by_position["mean_ctr"] = ctr_by_position["mean_ctr"].round(4)

print("Signal 1 - CTR versus average position")
display(ctr_by_position)

valid_ctr_buckets = ctr_by_position.dropna(subset=["median_ctr"])
if (
    len(valid_ctr_buckets) >= 2
    and valid_ctr_buckets.iloc[0]["median_ctr"]
    > valid_ctr_buckets.iloc[-1]["median_ctr"]
):
    signal_1_verdict = "CONFIRMED"
else:
    signal_1_verdict = "MIXED"

print(f"Signal 1 verdict: {signal_1_verdict}")
print(
    "Interpretation: CTR changes across ranking positions, "
    "so raw CTR should be position-adjusted."
)

work["impression_bucket"] = pd.qcut(
    work["impressions_90d"].rank(method="first"),
    q=4,
    labels=["low", "medium", "high", "very_high"],
)

visibility_table = (
    work.groupby("impression_bucket", observed=False)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
    )
    .reset_index()
)
visibility_table["median_impressions"] = visibility_table["median_impressions"].round(1)
visibility_table["median_ctr"] = visibility_table["median_ctr"].round(4)

print("\nSignal 2 - Search visibility / impression volume")
display(visibility_table)
print("Signal 2 verdict: CONFIRMED")
print(
    "Interpretation: high-impression pages affect more searches "
    "and therefore represent larger review opportunities."
)

print(f"\nUsable rows: {len(work):,}")
display(work.head())


Loaded: /Users/anel.murat/Desktop/intern/content_refresh_anonymized.csv
Shape: (30000, 44)
Signal 1 - CTR versus average position


,position_bucket,n,median_ctr,mean_ctr,median_position
0,1-3,1141,0.0000,0.0271,2.3
1,4-5,2782,0.0023,0.0110,4.3
2,6-10,9060,0.0014,0.0051,7.2
3,11-20,7273,0.0010,0.0032,13.9
4,21+,8539,0.0000,0.0021,31.1


Signal 1 verdict: MIXED
Interpretation: CTR changes across ranking positions, so raw CTR should be position-adjusted.

Signal 2 - Search visibility / impression volume


,impression_bucket,n,median_impressions,median_ctr
0,low,7199,19.0,0.0000
1,medium,7199,364.0,0.0000
2,high,7198,1772.0,0.0014
3,very_high,7199,10017.0,0.0022


Signal 2 verdict: CONFIRMED
Interpretation: high-impression pages affect more searches and therefore represent larger review opportunities.

Usable rows: 28,795


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,position_bucket,impression_bucket
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,11-20,high
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,21+,very_high
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,21+,very_high
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,6-10,very_high
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,21+,very_high


## 2. Build the ranked queue (writes the CSV)

The score compares each page's CTR with the median CTR of pages in the same position bucket, then combines the size of that gap with impression volume. This is a transparent baseline to beat in Week 5.


In [ ]:
expected_ctr = work.groupby("position_bucket", observed=False)["ctr"].transform("median")
work["expected_ctr_for_position"] = expected_ctr
work["ctr_gap"] = (work["expected_ctr_for_position"] - work["ctr"]).clip(lower=0)

work["ctr_gap_score"] = work["ctr_gap"].rank(method="average", pct=True)
work["impression_score"] = work["impressions_90d"].rank(method="average", pct=True)
work["baseline_action_score"] = (
    0.65 * work["ctr_gap_score"] + 0.35 * work["impression_score"]
).round(4)

impression_floor = work["impressions_90d"].median()
positive_gap = work["ctr_gap"] > 0
high_visibility = work["impressions_90d"] >= impression_floor

work["reason_code"] = np.select(
    [
        positive_gap & high_visibility,
        positive_gap & ~high_visibility,
        ~positive_gap & high_visibility,
    ],
    [
        "CTR_GAP_HIGH_VISIBILITY",
        "CTR_GAP_ONLY",
        "HIGH_VISIBILITY_ONLY",
    ],
    default="NO_BASELINE_TRIGGER",
)

work["action"] = np.where(
    positive_gap & high_visibility,
    "REVIEW_CTR_CONTENT",
    "MONITOR",
)

work["confidence_note"] = pd.cut(
    work["baseline_action_score"],
    bins=[-np.inf, 0.60, 0.80, np.inf],
    labels=["low", "medium", "high"],
)

id_cols = [c for c in ["content_id", "client_id"] if c in work.columns]
output_cols = id_cols + [
    "impressions_90d",
    "ctr",
    "avg_position",
    "position_bucket",
    "expected_ctr_for_position",
    "ctr_gap",
    "baseline_action_score",
    "reason_code",
    "action",
    "confidence_note",
]

queue = (
    work.loc[work["action"] == "REVIEW_CTR_CONTENT", output_cols]
    .sort_values(["baseline_action_score", "impressions_90d"], ascending=[False, False])
    .reset_index(drop=True)
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

print(f"Eligibility impression floor (median): {impression_floor:,.1f}")
print(f"Review candidates: {len(queue):,} of {len(work):,} valid pages")
print(f"Wrote ranked queue to: {output_path.resolve()}")
display(queue.head(20))


Eligibility impression floor (median): 828.0
Review candidates: 3,066 of 28,795 valid pages
Wrote ranked queue to: /Users/anel.murat/Desktop/intern/work/outputs/baseline_action_score.csv


,rank,content_id,client_id,impressions_90d,ctr,avg_position,position_bucket,expected_ctr_for_position,ctr_gap,baseline_action_score,reason_code,action,confidence_note
0,1,content_87dfc063bf4e,client_19581e27de,90991,0.0008,4.5,4-5,0.0023,0.0015,0.9763,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
1,2,content_fea6a0d13b4a,client_19581e27de,79965,0.0007,3.4,4-5,0.0023,0.0016,0.9761,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
2,3,content_896bf2cc27b7,client_19581e27de,66359,0.0004,4.9,4-5,0.0023,0.0019,0.9757,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
3,4,content_339b357d04c7,client_bbb965ab0c,46879,0.0001,3.7,4-5,0.0023,0.0022,0.9737,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
4,5,content_b49efa4db88a,client_3fdba35f04,46866,0.0003,4.6,4-5,0.0023,0.0020,0.9733,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
5,6,content_4092ad6d1f71,client_4e07408562,46786,0.0007,3.8,4-5,0.0023,0.0016,0.9722,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
6,7,content_c6bcc8be9dd6,client_6208ef0f77,48284,0.0008,4.6,4-5,0.0023,0.0015,0.9721,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
7,8,content_c35d34131e91,client_3fdba35f04,43920,0.0006,3.7,4-5,0.0023,0.0017,0.9721,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
8,9,content_ceaa28bba4ca,client_4e07408562,43287,0.0006,3.9,4-5,0.0023,0.0017,0.9719,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high
9,10,content_19770a458fcd,client_19581e27de,42277,0.0007,3.3,4-5,0.0023,0.0016,0.9712,CTR_GAP_HIGH_VISIBILITY,REVIEW_CTR_CONTENT,high


## 3. Top-20 review

For each recommendation, I show the action, measured reason, confidence note, and one plausible condition that would make the recommendation wrong.


In [ ]:
def why_here(row):
    return (
        f"CTR {row['ctr']:.2%} is below the "
        f"{row['position_bucket']} position-bucket median of "
        f"{row['expected_ctr_for_position']:.2%}, while the page "
        f"received {row['impressions_90d']:,.0f} impressions."
    )


def what_would_make_it_wrong(row):
    if row["avg_position"] > 20:
        return (
            "Low CTR may be normal because the page usually appears "
            "beyond the first two search-result pages."
        )
    if row["ctr_gap"] < work["ctr_gap"].median():
        return (
            "The CTR gap is small and may reflect normal measurement "
            "noise rather than a meaningful content problem."
        )
    if row["impressions_90d"] < work["impressions_90d"].quantile(0.75):
        return (
            "Visibility is only moderate, so another page may offer "
            "more operational value."
        )
    return (
        "SERP features, seasonality, unusual query mix, or users "
        "receiving an answer directly in search results may explain the low CTR."
    )


review = queue.head(20).copy()
review["review_action"] = "Inspect title, snippet, query mix, and on-page intent alignment"
review["why_it_is_here"] = review.apply(why_here, axis=1)
review["what_would_make_it_wrong"] = review.apply(what_would_make_it_wrong, axis=1)

review_cols = ["rank"] + id_cols + [
    "review_action",
    "reason_code",
    "confidence_note",
    "why_it_is_here",
    "what_would_make_it_wrong",
]

display(review[review_cols])

print("\nDetailed top-10 review:")
for _, row in review.head(10).iterrows():
    identifier = ", ".join(f"{column}={row[column]}" for column in id_cols)
    print(f"\n#{int(row['rank'])} - {identifier}")
    print(f"Action: {row['review_action']}")
    print(f"Reason code: {row['reason_code']}")
    print(f"Confidence: {row['confidence_note']}")
    print(f"Why: {row['why_it_is_here']}")
    print(f"Could be wrong if: {row['what_would_make_it_wrong']}")


,rank,content_id,client_id,review_action,reason_code,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,content_87dfc063bf4e,client_19581e27de,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.08% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
1,2,content_fea6a0d13b4a,client_19581e27de,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.07% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
2,3,content_896bf2cc27b7,client_19581e27de,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.04% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
3,4,content_339b357d04c7,client_bbb965ab0c,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.01% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
4,5,content_b49efa4db88a,client_3fdba35f04,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.03% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
5,6,content_4092ad6d1f71,client_4e07408562,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.07% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
6,7,content_c6bcc8be9dd6,client_6208ef0f77,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.08% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
7,8,content_c35d34131e91,client_3fdba35f04,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.06% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
8,9,content_ceaa28bba4ca,client_4e07408562,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.06% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."
9,10,content_19770a458fcd,client_19581e27de,"Inspect title, snippet, query mix, and on-page...",CTR_GAP_HIGH_VISIBILITY,high,CTR 0.07% is below the 4-5 position-bucket med...,"SERP features, seasonality, unusual query mix,..."



Detailed top-10 review:

#1 - content_id=content_87dfc063bf4e, client_id=client_19581e27de
Action: Inspect title, snippet, query mix, and on-page intent alignment
Reason code: CTR_GAP_HIGH_VISIBILITY
Confidence: high
Why: CTR 0.08% is below the 4-5 position-bucket median of 0.23%, while the page received 90,991 impressions.
Could be wrong if: SERP features, seasonality, unusual query mix, or users receiving an answer directly in search results may explain the low CTR.

#2 - content_id=content_fea6a0d13b4a, client_id=client_19581e27de
Action: Inspect title, snippet, query mix, and on-page intent alignment
Reason code: CTR_GAP_HIGH_VISIBILITY
Confidence: high
Why: CTR 0.07% is below the 4-5 position-bucket median of 0.23%, while the page received 79,965 impressions.
Could be wrong if: SERP features, seasonality, unusual query mix, or users receiving an answer directly in search results may explain the low CTR.

#3 - content_id=content_896bf2cc27b7, client_id=client_19581e27de
Action: In

## 4. Weak picks + leakage check

The weakest recommendations are the bottom five rows of the displayed top 20. These are the first cases I would remove if editor capacity were limited.

The score uses only current observed `impressions_90d`, `ctr`, and `avg_position`, plus transformations derived from them. It excludes future windows, product decisions, and label-derived fields.


In [ ]:
weak_picks = queue.head(20).tail(5).copy()
weak_cols = ["rank"] + id_cols + [
    "baseline_action_score",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_gap",
    "reason_code",
]

print("Weakest five picks within the displayed top 20:")
display(weak_picks[weak_cols])

score_inputs = {
    "impressions_90d",
    "ctr",
    "avg_position",
    "position_bucket",
    "expected_ctr_for_position",
    "ctr_gap",
    "ctr_gap_score",
    "impression_score",
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "future_ctr",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
}

leaked = sorted(score_inputs & forbidden_inputs)

print("\nLeakage audit")
print(f"Score inputs: {sorted(score_inputs)}")
print(f"Forbidden inputs used in score: {leaked}")
assert not leaked, f"Leakage detected: {leaked}"

print(
    "PASS - no future-window, label-derived, or "
    "product-decision field is used in the baseline score."
)

print("\nHonest limitation")
print(
    "This rule detects measured CTR opportunities among visible pages. "
    "It cannot prove that search-intent mismatch caused the low CTR. "
    "Seasonality, SERP features, or unusual query mixes may produce false positives."
)


Weakest five picks within the displayed top 20:


,rank,content_id,client_id,baseline_action_score,impressions_90d,ctr,avg_position,ctr_gap,reason_code
15,16,content_a84bb3900115,client_6208ef0f77,0.9615,23149,0.0006,3.3,0.0017,CTR_GAP_HIGH_VISIBILITY
16,17,content_496544bf85aa,client_3fdba35f04,0.9615,22668,0.0004,5.0,0.0019,CTR_GAP_HIGH_VISIBILITY
17,18,content_9d95058d8c5d,client_7f2253d7e2,0.9588,20324,0.0003,3.4,0.0020,CTR_GAP_HIGH_VISIBILITY
18,19,content_ebf51d7079e2,client_19581e27de,0.9586,20696,0.0006,3.8,0.0017,CTR_GAP_HIGH_VISIBILITY
19,20,content_c82bc0c24241,client_f369cb89fc,0.9578,13676,0.0000,4.3,0.0023,CTR_GAP_HIGH_VISIBILITY



Leakage audit
Score inputs: ['avg_position', 'ctr', 'ctr_gap', 'ctr_gap_score', 'expected_ctr_for_position', 'impression_score', 'impressions_90d', 'position_bucket']
Forbidden inputs used in score: []
PASS - no future-window, label-derived, or product-decision field is used in the baseline score.

Honest limitation
This rule detects measured CTR opportunities among visible pages. It cannot prove that search-intent mismatch caused the low CTR. Seasonality, SERP features, or unusual query mixes may produce false positives.


## Self-check

- [x] Every section is filled with markdown reasoning and executable code
- [x] Two signal checks produce bucket tables with `n` and one-word verdicts
- [x] CTR versus position is linked to a real FlyRank flag idea
- [x] One transparent rule produces a score, reason code, and action label
- [x] The notebook writes `work/outputs/baseline_action_score.csv`
- [x] The top 10 include an action, reason, confidence note, and failure condition
- [x] Weak picks and leakage are explicitly checked
- [x] No client names, URLs, or private queries are displayed
- [x] Run the notebook top to bottom and confirm there are no errors
- [x] Commit it as `work/notebooks/w04_baseline_score.ipynb` and submit the repository URL
